# Brazil — MLP Regressor

## 1. Imports e configurações

In [1]:
from pathlib import Path
import json, warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")
CSV_PATH=Path("datasets/brazil.csv")
TARGET="goals_brazil"
TEST_SIZE=0.20
RANDOM_STATE=42

from sklearn.neural_network import MLPRegressor

ModuleNotFoundError: No module named 'joblib'

## 2. Limpeza

In [ ]:
def load_and_clean(path):
    df=pd.read_csv(path).copy()
    df["date"]=pd.to_datetime(df["Date"],format="%d %b %Y",errors="coerce")
    df=df.sort_values("date",kind="stable").drop_duplicates("date",keep="first").reset_index(drop=True)
    score=df["Score"].astype("string").str.extract(r"^\s*(\d+)\s*-\s*(\d+)")
    df["home_goals"]=pd.to_numeric(score[0],errors="coerce")
    df["away_goals"]=pd.to_numeric(score[1],errors="coerce")
    df["is_home"]=df["Match"].str.startswith("Brazil v ").astype(int)
    df["opponent"]=np.where(df["is_home"].eq(1),df["Match"].str.replace(r"^Brazil v ","",regex=True),df["Match"].str.replace(r" v Brazil$","",regex=True))
    df["goals_brazil"]=np.where(df["is_home"].eq(1),df["home_goals"],df["away_goals"])
    df["goals_against"]=np.where(df["is_home"].eq(1),df["away_goals"],df["home_goals"])
    df["points"]=df["Result"].map({"W":3,"D":1,"L":0})
    df=df.dropna(subset=["date","Competition","opponent","goals_brazil","goals_against","points"]).copy()
    for c in ["goals_brazil","goals_against","points"]: df[c]=df[c].astype(float)
    return df.reset_index(drop=True)
raw=load_and_clean(CSV_PATH)
print("Linhas após limpeza:",len(raw)); display(raw.head())


## 3. Feature Engineering sem vazamento

In [ ]:
def create_features(df):
    data=df.copy(); data["days_since_last_match"]=data["date"].diff().dt.days.clip(lower=0)
    lags=[1,2,3,5,10]
    for lag in lags:
        data[f"goals_lag_{lag}"]=data["goals_brazil"].shift(lag)
        data[f"conceded_lag_{lag}"]=data["goals_against"].shift(lag)
        data[f"points_lag_{lag}"]=data["points"].shift(lag)
    windows=[3,5,10,20]
    g=data["goals_brazil"].shift(1); ga=data["goals_against"].shift(1); pts=data["points"].shift(1)
    for w in windows:
        data[f"goals_roll_mean_{w}"]=g.rolling(w).mean(); data[f"goals_roll_std_{w}"]=g.rolling(w).std()
        data[f"conceded_roll_mean_{w}"]=ga.rolling(w).mean(); data[f"points_roll_mean_{w}"]=pts.rolling(w).mean()
    data["month"]=data["date"].dt.month; data["month_sin"]=np.sin(2*np.pi*data["month"]/12); data["month_cos"]=np.cos(2*np.pi*data["month"]/12)
    data["day_of_year"]=data["date"].dt.dayofyear; data["doy_sin"]=np.sin(2*np.pi*data["day_of_year"]/365.25); data["doy_cos"]=np.cos(2*np.pi*data["day_of_year"]/365.25)
    data["year"]=data["date"].dt.year
    categorical_features=["opponent","Competition"]
    numeric_features=["is_home","days_since_last_match","month_sin","month_cos","doy_sin","doy_cos","year"]
    for lag in lags: numeric_features += [f"goals_lag_{lag}",f"conceded_lag_{lag}",f"points_lag_{lag}"]
    for w in windows: numeric_features += [f"goals_roll_mean_{w}",f"goals_roll_std_{w}",f"conceded_roll_mean_{w}",f"points_roll_mean_{w}"]
    feature_columns=numeric_features+categorical_features
    data=data.dropna(subset=feature_columns+[TARGET]).reset_index(drop=True)
    return data,numeric_features,categorical_features,feature_columns

data,numeric_features,categorical_features,feature_columns=create_features(raw)
split_idx=int(len(data)*(1-TEST_SIZE)); train=data.iloc[:split_idx].copy(); test=data.iloc[split_idx:].copy()
X_train=train[feature_columns]; y_train=train[TARGET]; X_test=test[feature_columns]; y_test=test[TARGET]
print("Treino:",len(train),"Teste:",len(test))


## 4. Pré-processamento

In [ ]:
def build_preprocessor():
    num=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
    cat=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
    return ColumnTransformer([("numeric",num,numeric_features),("categorical",cat,categorical_features)])


## 5. Modelo e tuning temporal

In [ ]:
pipeline=Pipeline([("preprocess",build_preprocessor()),("model",MLPRegressor(random_state=RANDOM_STATE,max_iter=1500,early_stopping=False))])
param_distributions={'model__hidden_layer_sizes': [(32,), (64,), (128,), (64, 32), (128, 64), (128, 64, 32)], 'model__activation': ['relu', 'tanh'], 'model__alpha': [0.0001, 0.001, 0.01, 0.1], 'model__learning_rate_init': [0.0005, 0.001, 0.005, 0.01]}
cv=TimeSeriesSplit(n_splits=5)
search=RandomizedSearchCV(pipeline,param_distributions,n_iter=min(12,100),scoring="neg_mean_absolute_error",cv=cv,random_state=RANDOM_STATE,n_jobs=-1,refit=True,verbose=1)
search.fit(X_train,y_train)
print("Melhores hiperparâmetros:",search.best_params_)
print("MAE médio validação temporal:",round(-search.best_score_,4))


## 6. Walk-forward final

In [ ]:
def walk_forward(best_pipeline):
    preds=[]
    for i in range(len(X_test)):
        Xh=pd.concat([X_train,X_test.iloc[:i]],axis=0); yh=pd.concat([y_train,y_test.iloc[:i]],axis=0)
        m=clone(best_pipeline); m.fit(Xh,yh); preds.append(float(np.ravel(m.predict(X_test.iloc[[i]]))[0]))
    preds=np.asarray(preds)
    out=pd.DataFrame({"date":test["date"].values,"real":y_test.values,"prediction":preds,"absolute_error":np.abs(y_test.values-preds)})
    return mean_absolute_error(y_test,preds),out


In [ ]:
final_mae,predictions=walk_forward(search.best_estimator_)
print("MAE final walk-forward:",round(final_mae,4))
display(predictions.head(10))

## 7. Treino final e salvamento

In [ ]:
OUTPUT_DIR=Path("outputs_05_mlp_brazil"); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
final_model=clone(search.best_estimator_); final_model.fit(data[feature_columns],data[TARGET])
joblib.dump(final_model,OUTPUT_DIR/"modelo_final.joblib")
predictions.to_csv(OUTPUT_DIR/"walk_forward_predictions.csv",index=False)
meta={"model":"MLP Regressor","target":TARGET,"horizon":"1 partida","MAE_walk_forward":float(final_mae),"CV_MAE":float(-search.best_score_),"best_params":search.best_params_}
with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2,default=str)
print("Salvo em:",OUTPUT_DIR.resolve())
